# Effective op time analysis

In [1]:
import os
import csv
import pandas as pd
import plotly.express as px

with open("../Data and descriptions/Case Rigshospitalet - Cancelled operations.csv", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

['Case-ID Anonymous;Dato og tid;Primær procedure - Tekst;Aflyst efter operationsprogrammet er afsluttet?;"Aflyst på dagen for operationen? ";Aflysningsårsag;Forventet varighed (min.);Ombooket;Stue;Operationsgang ID']
['7574;2024-01-03 22:05:00', '000;"GASTROSKOPI";"Nej";"Nej";"";93;"Nej";"CKO LEJE 01";"618"']
['7868;2024-01-05 20:50:00', '000;"KOLOSKOPI";"Nej";"Nej";"";87;"Nej";"CKO LEJE 03";"618"']
['8054;2024-01-07 14:10:00', '000;"INVR EMBOLISERING";"Nej";"Nej";"";132;"Nej";"CKO RTG LEJE U/A";"618"']
['11481;2024-01-26 11:00:00', '000;"INVR EMBOLISERING";"Nej";"Nej";"";207;"Nej";"CKO RTG LEJE 14";"618"']
['16394;2024-02-26 08:05:00', '000;"INVR SCLEROSERING";"Nej";"Nej";"";111;"Nej";"CKO RTG LEJE 11";"618"']
['19035;2024-03-11 02:20:00', '000;"GASTROSKOPI";"Nej";"Nej";"";92;"Nej";"CKO LEJE 01";"618"']
['26827;2024-04-26 12:00:00', '000;"RESEKTION TYNDTARM";"Nej";"Nej";"";215;"Nej";"CKO LEJE 02";"618"']
['27141;2024-04-27 11:00:00', '000;"INVR PTC/DRÆNAGE";"Nej";"Nej";"";126;"Nej";"C

In [2]:
df_cancelled = pd.read_csv("../Data and descriptions/Case Rigshospitalet - Cancelled operations.csv", sep=';')
df_complete = pd.read_csv("../Data and descriptions/Case Rigshospitalet - Completed operations.csv", sep = ';')

C:\Users\frida\AppData\Local\Temp\ipykernel_30200\3600004165.py:2: DtypeWarning: Columns (13,14,21,22,23,24,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df_complete = pd.read_csv("../Data and descriptions/Case Rigshospitalet - Completed operations.csv", sep = ';')


In [3]:
# remove columns with staff and resources
df_complete_wo = df_complete.drop(columns=[col for col in df_complete.columns if col.startswith("Ressource")])
df_complete_wo = df_complete_wo.drop(columns=[col for col in df_complete_wo.columns if col.startswith("Staff")])
#df_complete_wo.describe()

In [4]:
df_complete_wo = df_complete_wo.drop(columns=['Forsinkelsesårsag'])

In [5]:
datetime_variables = ['Dato', 'Pt ankommet til hospitalet', 'Planlagt stue klargøring start', 'Planlagt stue klargøring start', 'Stue klargøring start', 'Stue klargjort', 'Patient på stuen', 'Patient på stuen (Planlagt)', 'Procedure start', 'Procedure slut', 'Patient klar til afgang', 'Patient forlader stuen (Planlagt)', 'Patient forlader stuen', 'Stue rengjort (Planlagt)', 'Stue rengøring start', 'Stue rengjort', 'Patient forlader afdeling']
# Make datetime objects
for i in datetime_variables:
    df_complete_wo[i] = pd.to_datetime(df_complete_wo[i], format='%Y-%m-%d %H:%M:%S,%f')

In [6]:
# Make datetime objects
for i in datetime_variables:
    df_complete_wo[i] = pd.to_datetime(df_complete_wo[i], format='%Y-%m-%d %H:%M:%S,%f')

In [7]:
df_complete_wo['Overskredet (minutter)'].max()

1560.0

In [8]:
df_complete_wo['Forsinkelse (minutter)'].max()

1585.0

In [9]:
df_no_outliers = df_complete_wo[(df_complete_wo['Forsinkelse (minutter)'] >= -300) & 
                  (df_complete_wo['Forsinkelse (minutter)'] <= 300)].copy()

In [10]:
# Udtræk 'Time på dagen' og 'Ugedag'
df_no_outliers['Time'] = df_no_outliers['Procedure start'].dt.hour
df_no_outliers['Ugedag'] = df_no_outliers['Procedure start'].dt.day_name()

# Sorter ugedage så de ikke står alfabetisk
dage_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_no_outliers['Ugedag'] = pd.Categorical(df_no_outliers['Ugedag'], categories=dage_orden, ordered=True)

In [11]:
op_talt_per_time = df_no_outliers[df_no_outliers['Akut case (J/N)'] == 'Nej']['Time'].value_counts().sort_index()
print("Antal planlagte operationer per time:")
print(op_talt_per_time)

Antal planlagte operationer per time:
Time
0.0         2
1.0         6
2.0         1
3.0         2
7.0         2
8.0     14750
9.0     26579
10.0    16149
11.0    14654
12.0    13015
13.0    12658
14.0     5792
15.0     1037
16.0      603
17.0      304
18.0      115
19.0       68
20.0       65
21.0       34
22.0       22
23.0        9
Name: count, dtype: int64


In [12]:
df_no_outliers.columns

Index(['Case-ID Anonymous', 'Patient Alder', 'Speciale', 'Stue',
       'Operationsgang ID', 'Akut case (J/N)', 'Dato',
       'Pt ankommet til hospitalet', 'Planlagt stue klargøring start',
       'Stue klargøring start', 'Stue klargjort',
       'Patient på stuen (Planlagt)', 'Patient på stuen', 'Anæstesistart',
       'Anæstesi melder klar', 'Procedure start', 'Procedure slut',
       'Patient klar til afgang', 'Patient forlader stuen (Planlagt)',
       'Patient forlader stuen', 'Stue rengjort (Planlagt)',
       'Stue rengøring start', 'Stue rengjort', 'I opvågning', 'Anæstesistop',
       'Klar til udskrivelse efter opvågning', 'Patient forlader afdeling',
       'Forsinkelse (minutter)', 'Overskredet (minutter)',
       'Procedure - Tekst & ID', 'Aktionsdiagnose - Kode & tekst',
       'Aktionsdiagnose - Gruppe', 'Time', 'Ugedag'],
      dtype='object')

In [13]:

df_eye = df_no_outliers[df_no_outliers['Speciale'] == 'Øjenkirurgi'].copy()

# Faktisk Varighed
df_eye['Faktisk_Varighed'] = (df_eye['Patient forlader stuen'] - df_eye['Patient på stuen']).dt.total_seconds() / 60

# Planlagt Varighed
# Vi bruger 'Patient forlader stuen (Planlagt)' og 'Patient på stuen (Planlagt)' 
# for at se, hvad der var afsat til hele seancen
df_eye['Planlagt_Varighed'] = (df_eye['Patient forlader stuen (Planlagt)'] - df_eye['Patient på stuen (Planlagt)']).dt.total_seconds() / 60

# Afvigelse
# Positiv værdi - Operationen tog længere tid 
# Negativ værdi - Operationen tog kortere tid
df_eye['Varighed_Afvigelse'] = df_eye['Faktisk_Varighed'] - df_eye['Planlagt_Varighed']

In [46]:
df_eye['Afvigelse2'] = df_eye['Overskredet (minutter)'] - df_eye['Forsinkelse (minutter)']

In [14]:
df_eye['Klargøring_afvigelse'] = (df_eye['Stue klargøring start'] - df_eye['Planlagt stue klargøring start']).dt.total_seconds() / 60

In [47]:
df_eye.columns

Index(['Case-ID Anonymous', 'Patient Alder', 'Speciale', 'Stue',
       'Operationsgang ID', 'Akut case (J/N)', 'Dato',
       'Pt ankommet til hospitalet', 'Planlagt stue klargøring start',
       'Stue klargøring start', 'Stue klargjort',
       'Patient på stuen (Planlagt)', 'Patient på stuen', 'Anæstesistart',
       'Anæstesi melder klar', 'Procedure start', 'Procedure slut',
       'Patient klar til afgang', 'Patient forlader stuen (Planlagt)',
       'Patient forlader stuen', 'Stue rengjort (Planlagt)',
       'Stue rengøring start', 'Stue rengjort', 'I opvågning', 'Anæstesistop',
       'Klar til udskrivelse efter opvågning', 'Patient forlader afdeling',
       'Forsinkelse (minutter)', 'Overskredet (minutter)',
       'Procedure - Tekst & ID', 'Aktionsdiagnose - Kode & tekst',
       'Aktionsdiagnose - Gruppe', 'Time', 'Ugedag', 'Faktisk_Varighed',
       'Planlagt_Varighed', 'Varighed_Afvigelse', 'Klargøring_afvigelse',
       'Afvigelse2'],
      dtype='object')

In [60]:
df_eye[['Patient på stuen (Planlagt)', 'Procedure start', 'Patient på stuen']]

,Patient på stuen (Planlagt),Procedure start,Patient på stuen
28,2024-10-22 11:30:00,2024-10-22 11:45:00,2024-10-22 11:33:00
39,2024-01-16 12:15:00,2024-01-16 13:02:00,2024-01-16 12:52:00
44,2024-01-24 13:00:00,2024-01-24 13:05:00,2024-01-24 12:50:00
58,2024-06-05 12:30:00,2024-06-05 13:35:00,2024-06-05 13:29:00
59,2024-06-05 12:30:00,2024-06-05 13:35:00,2024-06-05 13:29:00
...,...,...,...
133153,2025-12-18 12:30:00,2025-12-18 13:22:00,2025-12-18 13:17:00
133154,2025-12-23 11:00:00,2025-12-23 11:40:00,2025-12-23 11:35:00
133155,2025-12-19 11:00:00,2025-12-19 11:56:00,2025-12-19 11:52:00
133156,2025-12-22 08:30:00,2025-12-22 09:19:00,2025-12-22 09:12:00


# Nye ting

In [51]:
# 2. Beregn varigheden i minutter
df_eye['Faktisk_klargøringstid'] = (df_eye['Stue klargjort'] - df_eye['Stue klargøring start']).dt.total_seconds() / 60

# 3. Se gennemsnittet
gns_klargøring = df_eye['Faktisk_klargøringstid'].mean()
median_klargøring = df_eye['Faktisk_klargøringstid'].median()

print(f"Gennemsnitlig klargøringstid: {gns_klargøring:.2f} minutter")
print(f"Median klargøringstid: {median_klargøring:.2f} minutter")

Gennemsnitlig klargøringstid: 1.71 minutter
Median klargøringstid: 0.00 minutter


In [61]:
# Filtrer så vi kun ser på rækker med logiske tidsintervaller (f.eks. mellem 1 min og 120 min)
rensede_data_klargøring = df_eye[df_eye['Faktisk_klargøringstid'] > 0]['Faktisk_klargøringstid']
rensede_data_rengøring = df_eye[df_eye['Faktisk_rengøringstid'] > 0]['Faktisk_rengøringstid']

print(f" gns. klargøring: {rensede_data_klargøring.mean():.2f} min")
print(f" gns. rengøring: {rensede_data_rengøring.mean():.2f} min")

 gns. klargøring: 9.24 min
 gns. rengøring: 9.50 min


In [59]:
antal_negative = (df_eye['Faktisk_klargøringstid'] < 0).sum()
antal_negative

antal_ialt = (df_eye['Faktisk_klargøringstid']).sum()
antal_ialt

69628.0

In [54]:
# Beregn faktisk rengøringstid
df_eye['Faktisk_rengøringstid'] = (df_eye['Stue rengjort'] - df_eye['Stue rengøring start']).dt.total_seconds() / 60

# 3. Se gennemsnittet
gns_klargøring = df_eye['Faktisk_rengøringstid'].mean()
median_klargøring = df_eye['Faktisk_rengøringstid'].median()

print(f"Gennemsnitlig rengøringstid: {gns_klargøring:.2f} minutter")
print(f"Median rengøringstid: {median_klargøring:.2f} minutter")

Gennemsnitlig rengøringstid: 0.71 minutter
Median rengøringstid: 0.00 minutter


In [48]:
# Hvor stor en del af forsinkelsen skyldes klargøring vs. selve proceduren?
print(df_eye[['Klargøring_afvigelse', 'Varighed_Afvigelse', 'Forsinkelse (minutter)']].describe())

# Se korrelationen - hvad hænger stærkest sammen med 'Forsinkelse (minutter)'?
print(df_eye[['Afvigelse2', 'Klargøring_afvigelse', 'Varighed_Afvigelse', 'Forsinkelse (minutter)']].corr())

       Klargøring_afvigelse  Varighed_Afvigelse  Forsinkelse (minutter)
count          40762.000000        40823.000000            40824.000000
mean              33.153035            4.350440               41.006026
std               39.338249           18.017114               36.797693
min            -1423.000000         -146.000000             -274.000000
25%               10.000000           -2.000000               20.000000
50%               33.000000            3.000000               39.000000
75%               55.000000           10.000000               60.000000
max              291.000000          514.000000              299.000000
                        Afvigelse2  Klargøring_afvigelse  Varighed_Afvigelse  \
Afvigelse2                1.000000              0.012779            1.000000   
Klargøring_afvigelse      0.012779              1.000000            0.012779   
Varighed_Afvigelse        1.000000              0.012779            1.000000   
Forsinkelse (minutter)    0.0168

In [16]:

# timerne fra 7 til og med 22
df_filtered = df_eye[(df_eye['Time'] >= 7) & (df_eye['Time'] <= 22)]

# 2. Lav den gennemsnitlige beregning baseret på det filtrerede data
pivot_data = df_filtered.groupby('Time')[['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse']].mean().reset_index()

# 3. Lav plottet
fig = px.line(pivot_data, x='Time', y=['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse'],
              title='Forsinkelse vs. Operationseffektivitet (Kl. 07-22)',
              labels={'value': 'Minutter', 'Time': 'Time på dagen'},
              markers=True,
              template="plotly_white")

# Gør x-aksen pænere, så den viser hver time
fig.update_layout(xaxis=dict(tickmode='linear', tick0=7, dtick=1))

fig.show()

In [17]:

# Faktisk Varighed
df_no_outliers['Faktisk_Varighed'] = (df_no_outliers['Patient forlader stuen'] - df_no_outliers['Patient på stuen']).dt.total_seconds() / 60

# Planlagt Varighed
# Vi bruger 'Patient forlader stuen (Planlagt)' og 'Patient på stuen (Planlagt)' 
# for at se, hvad der var afsat til hele seancen
df_no_outliers['Planlagt_Varighed'] = (df_no_outliers['Patient forlader stuen (Planlagt)'] - df_no_outliers['Patient på stuen (Planlagt)']).dt.total_seconds() / 60

# Afvigelse
# Positiv værdi - Operationen tog længere tid 
# Negativ værdi - Operationen tog kortere tid
df_no_outliers['Varighed_Afvigelse'] = df_no_outliers['Faktisk_Varighed'] - df_no_outliers['Planlagt_Varighed']

In [18]:
df_no_outliers['Klargøring_afvigelse'] = (df_no_outliers['Stue klargøring start'] - df_no_outliers['Planlagt stue klargøring start']).dt.total_seconds() / 60

In [19]:

# forsinkelse indrul
df_no_outliers['Forsinkelse_indrul'] = (df_no_outliers['Patient på stuen'] - df_no_outliers['Stue klargjort']).dt.total_seconds() / 60


In [20]:

# timerne fra 7 til og med 22
df_filtered = df_no_outliers[(df_no_outliers['Time'] >= 7) & (df_no_outliers['Time'] <= 22)]

# 2. Lav den gennemsnitlige beregning baseret på det filtrerede data
pivot_data = df_filtered.groupby('Time')[['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse', 'Forsinkelse_indrul']].mean().reset_index()

# 3. Lav plottet
fig = px.line(pivot_data, x='Time', y=['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse', 'Forsinkelse_indrul'],
              title='Forsinkelse vs. Operationseffektivitet (Kl. 07-22)',
              labels={'value': 'Minutter', 'Time': 'Time på dagen'},
              markers=True,
              template="plotly_white")

# Gør x-aksen pænere, så den viser hver time
fig.update_layout(xaxis=dict(tickmode='linear', tick0=7, dtick=1))

fig.show()

In [21]:
df_planlagt = df_no_outliers[df_no_outliers['Akut case (J/N)'] == 'Nej'].copy()

In [22]:

# timerne fra 7 til og med 22
df_filtered = df_planlagt[(df_planlagt['Time'] >= 8) & (df_planlagt['Time'] <= 22)]

# 2. Lav den gennemsnitlige beregning baseret på det filtrerede data
pivot_data = df_filtered.groupby('Time')[['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse', 'Forsinkelse_indrul']].mean().reset_index()

# 3. Lav plottet
fig = px.line(pivot_data, x='Time', y=['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse', 'Forsinkelse_indrul'],
              title='Forsinkelse vs. Operationseffektivitet (Kl. 07-22)',
              labels={'value': 'Minutter', 'Time': 'Time på dagen'},
              markers=True,
              template="plotly_white")

# Gør x-aksen pænere, så den viser hver time
fig.update_layout(xaxis=dict(tickmode='linear', tick0=7, dtick=1))

fig.show()

In [23]:
df_planlagt['Stue']

0         OPNORD 23.240 (H)
1         OPNORD 13.114 (H)
2         OPNORD 13.114 (H)
3         OPNORD 13.110 (H)
4                OPNORD 301
                ...        
133153     GLO Ø 46 STUE 07
133154     GLO Ø 46 STUE 07
133155     GLO Ø 36 STUE 05
133156     GLO Ø 36 STUE 05
133157            GLO VB 10
Name: Stue, Length: 106026, dtype: object

In [24]:
df_planlagt_eye = df_eye[df_eye['Akut case (J/N)'] == 'Nej'].copy()

In [25]:

# timerne fra 7 til og med 22
df_filtered = df_planlagt_eye[(df_planlagt_eye['Time'] >= 8) & (df_planlagt_eye['Time'] <= 22)]

# 2. Lav den gennemsnitlige beregning baseret på det filtrerede data
pivot_data = df_filtered.groupby('Time')[['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse']].mean().reset_index()

# 3. Lav plottet
fig = px.line(pivot_data, x='Time', y=['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse'],
              title='Forsinkelse vs. Operationseffektivitet (Kl. 07-22)',
              labels={'value': 'Minutter', 'Time': 'Time på dagen'},
              markers=True,
              template="plotly_white")

# Gør x-aksen pænere, så den viser hver time
fig.update_layout(xaxis=dict(tickmode='linear', tick0=7, dtick=1))

fig.show()

In [44]:
df_planlagt_eye


,Case-ID Anonymous,Patient Alder,Speciale,Stue,Operationsgang ID,Akut case (J/N),Dato,Pt ankommet til hospitalet,Planlagt stue klargøring start,Stue klargøring start,...,Overskredet (minutter),Procedure - Tekst & ID,Aktionsdiagnose - Kode & tekst,Aktionsdiagnose - Gruppe,Time,Ugedag,Faktisk_Varighed,Planlagt_Varighed,Varighed_Afvigelse,Klargøring_afvigelse
28,271,85,Øjenkirurgi,GLO Ø 36 STUE 01,625,Nej,2024-10-22,2024-10-22 10:11:00,2024-10-22 11:30:00,2024-10-22 11:31:00,...,2.0,"PTOSE, LEVATORKIRURGI - LA [1070010580]",DH024: Blefaroptose,"Sygdomme i øjenlåg, tårekirtel og øjenhule",11.0,Tuesday,59.0,60.0,-1.0,1.0
39,349,68,Øjenkirurgi,GLO Ø 36 STUE 01,625,Nej,2024-01-16,2024-01-16 11:57:00,2024-01-16 12:15:00,2024-01-16 12:43:00,...,45.0,"EKTROPION MED RETRAKTOR-, MEDIAL-, OG LATERAL...",DH109: Konjunktivitis UNS,Sygdomme i øjets bindehinde,13.0,Tuesday,63.0,55.0,8.0,28.0
44,449,74,Øjenkirurgi,GLO Ø 36 STUE 03,625,Nej,2024-01-24,2024-01-24 10:02:00,2024-01-24 13:00:00,2024-01-24 12:45:00,...,21.0,"BLEPHAROPLASTIK, ØVRE BILATERAL - LA [1070010466]",DH024: Blefaroptose,"Sygdomme i øjenlåg, tårekirtel og øjenhule",13.0,Wednesday,81.0,50.0,31.0,-15.0
58,476,75,Øjenkirurgi,GLO Ø 36 STUE 01,625,Nej,2024-06-05,2024-06-05 12:10:00,2024-06-05 12:30:00,2024-06-05 13:28:00,...,71.0,"INTUBATION AF TÅREKANAL TIL TÅRESÆK, UNILATERA...",DH042: Epifora,"Sygdomme i øjenlåg, tårekirtel og øjenhule",13.0,Wednesday,40.0,28.0,12.0,58.0
59,476,75,Øjenkirurgi,GLO Ø 36 STUE 01,625,Nej,2024-06-05,2024-06-05 12:10:00,2024-06-05 12:30:00,2024-06-05 13:28:00,...,71.0,"SONDERING AF TÅREKANAL, BILATERAL - LA [107001...",DH042: Epifora,"Sygdomme i øjenlåg, tårekirtel og øjenhule",13.0,Wednesday,40.0,28.0,12.0,58.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133153,120283,59,Øjenkirurgi,GLO Ø 46 STUE 07,625,Nej,2025-12-18,2025-12-18 12:09:00,2025-12-18 12:30:00,2025-12-18 13:13:00,...,51.0,FAKO 2 - LA [1070010663],DH259: Aldersbetinget grå stær (>=50 år) UNS,Sygdomme i øjets linse,13.0,Thursday,19.0,15.0,4.0,43.0
133154,120284,58,Øjenkirurgi,GLO Ø 46 STUE 07,625,Nej,2025-12-23,2025-12-23 10:50:00,2025-12-23 11:00:00,2025-12-23 11:32:00,...,40.0,FAKO 3 - LA [1070010664],DH259: Aldersbetinget grå stær (>=50 år) UNS,Sygdomme i øjets linse,11.0,Tuesday,20.0,15.0,5.0,32.0
133155,120291,70,Øjenkirurgi,GLO Ø 36 STUE 05,625,Nej,2025-12-19,2025-12-19 10:39:00,2025-12-19 11:00:00,2025-12-19 11:52:00,...,52.0,FAKO 3 - LA [1070010664],DH259: Aldersbetinget grå stær (>=50 år) UNS,Sygdomme i øjets linse,11.0,Friday,15.0,15.0,0.0,52.0
133156,120337,87,Øjenkirurgi,GLO Ø 36 STUE 05,625,Nej,2025-12-22,2025-12-22 08:17:00,2025-12-22 08:30:00,2025-12-22 09:04:00,...,50.0,FAKO - LA BILATERAL [1070015679],DH259: Aldersbetinget grå stær (>=50 år) UNS,Sygdomme i øjets linse,9.0,Monday,38.0,30.0,8.0,34.0


In [43]:
df_planlagt_eye[['Planlagt stue klargøring start', 'Stue klargøring start', 'Procedure start', 'Stue rengjort (Planlagt)', 'Stue rengjort']]

,Planlagt stue klargøring start,Stue klargøring start,Procedure start,Stue rengjort (Planlagt),Stue rengjort
28,2024-10-22 11:30:00,2024-10-22 11:31:00,2024-10-22 11:45:00,2024-10-22 12:30:00,NaT
39,2024-01-16 12:15:00,2024-01-16 12:43:00,2024-01-16 13:02:00,2024-01-16 13:10:00,NaT
44,2024-01-24 13:00:00,2024-01-24 12:45:00,2024-01-24 13:05:00,2024-01-24 13:50:00,NaT
58,2024-06-05 12:30:00,2024-06-05 13:28:00,2024-06-05 13:35:00,2024-06-05 12:58:00,NaT
59,2024-06-05 12:30:00,2024-06-05 13:28:00,2024-06-05 13:35:00,2024-06-05 12:58:00,NaT
...,...,...,...,...,...
133153,2025-12-18 12:30:00,2025-12-18 13:13:00,2025-12-18 13:22:00,2025-12-18 12:45:00,2025-12-18 13:38:00
133154,2025-12-23 11:00:00,2025-12-23 11:32:00,2025-12-23 11:40:00,2025-12-23 11:15:00,2025-12-23 11:55:00
133155,2025-12-19 11:00:00,2025-12-19 11:52:00,2025-12-19 11:56:00,2025-12-19 11:15:00,2025-12-19 12:07:00
133156,2025-12-22 08:30:00,2025-12-22 09:04:00,2025-12-22 09:19:00,2025-12-22 09:00:00,2025-12-22 09:52:00


In [26]:
# Giver dig både min, max, gennemsnit og median for kl. 8
print(df_planlagt[df_planlagt['Time'] == 8]['Forsinkelse (minutter)'].describe())

count    14750.000000
mean        10.951186
std         20.318120
min       -299.000000
25%          2.000000
50%          9.000000
75%         22.000000
max        159.000000
Name: Forsinkelse (minutter), dtype: float64


In [27]:
# 1. Definer hvilken stue du vil se på (kopier navnet præcis fra dine data)
valgt_stue = 'GLO Ø 46 STUE 07' 

# 2. Filtrer for både tid OG den specifikke stue
df_filtered = df_planlagt[
    (df_planlagt['Time'] >= 8) & 
    (df_planlagt['Time'] <= 22) & 
    (df_planlagt['Stue'] == valgt_stue)
]

# 3. Lav den gennemsnitlige beregning (samme som før)
pivot_data = df_filtered.groupby('Time')[['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse', 'Forsinkelse_indrul']].mean().reset_index()

# 4. Lav plottet (opdateret titel så man kan se hvilken stue det er)
fig = px.line(pivot_data, x='Time', y=['Forsinkelse (minutter)', 'Varighed_Afvigelse', 'Klargøring_afvigelse', 'Forsinkelse_indrul'],
              title=f'Forsinkelse og afvigelser for {valgt_stue}',
              labels={'value': 'Minutter', 'Time': 'Time på dagen'},
              markers=True,
              template="plotly_white")

fig.update_layout(xaxis=dict(tickmode='linear', tick0=8, dtick=1))

fig.show()

In [28]:
# 1. Vælg den stue du vil undersøge
valgt_stue = 'GLO Ø 46 STUE 07'

# 2. Filtrer for både kl. 8 OG den specifikke stue
stats_kl8_stue = df_planlagt[
    (df_planlagt['Time'] == 8) & 
    (df_planlagt['Stue'] == valgt_stue)
]['Forsinkelse (minutter)'].describe()

print(f"Statistik for {valgt_stue} kl. 08:00:")
print(stats_kl8_stue)

Statistik for GLO Ø 46 STUE 07 kl. 08:00:
count    283.000000
mean      32.261484
std       10.807905
min      -65.000000
25%       31.000000
50%       34.000000
75%       37.000000
max       56.000000
Name: Forsinkelse (minutter), dtype: float64


In [29]:
df_planlagt['Stue'].unique()

array(['OPNORD 23.240 (H)', 'OPNORD 13.114 (H)', 'OPNORD 13.110 (H)',
       'OPNORD 301', 'RH TMK AMB 1', 'RH TMK AMB 2', 'OPNORD 13.109 (H)',
       'RH DAGKIR A (H)', 'OPNORD 302', 'RH DAGKIR D (H)',
       'GLO Ø 36 STUE 01', 'OPNORD 13.111 (H)', 'OPNORD 303',
       'GLO Ø 36 STUE 03', 'RH TMK AMB 3', 'GLO Ø 36 STUE 02',
       'OPNORD 13.116 (H)', 'OPNORD 13.106 (H)', 'RH TMK AMB 4',
       'RH DAGKIR B (H)', 'GLO VB 16', 'OPNORD 13.115 (H)',
       'RH DAGKIR C (H)', 'RH TMK AMB 5', 'GLO VB 14', 'RH DAGKIR E (H)',
       'GLO VB 15', 'CKO 3111', 'GLO Ø 46 STUE 09', 'GLO Ø 36 STUE 04',
       'GLO Ø 36 STUE 05', 'CKO LEJE 05', 'RT GN', 'GLO MB 08',
       'AMBULANT OPERATION 8067', 'GLO Ø 46 STUE 11',
       'CKO TRAUME RTG LEJE 13', 'GLO Ø 46 STUE 10', 'OPNORD 13.113 (H)',
       'OPNORD 23.244 (H)', 'OPNORD 13.105 (H)', 'GLO VB 10',
       'GLO Ø 46 STUE 07', 'OPNORD 13.112 (H)', 'CKO LEJE 04',
       'GLO Ø 46 STUE 08', 'OPNORD 13.103 (H)', 'CKO LEJE 03',
       'GLO Ø SEC-HK'

In [30]:
# 1. Lav en liste til at gemme resultaterne
stuer_med_positiv_min = []

# 2. Gå igennem alle unikke stuer
for stue in df_planlagt['Stue'].unique():
    # Filtrer data for den specifikke stue kl. 8
    subset = df_planlagt[(df_planlagt['Stue'] == stue) & (df_planlagt['Time'] == 8)]
    
    # Tjek om der overhovedet er data for denne stue kl. 8
    if not subset.empty:
        min_forsinkelse = subset['Forsinkelse (minutter)'].min()
        
        # 3. Gem stuen, hvis minimumsforsinkelsen er større end 0
        if min_forsinkelse > 0:
            stuer_med_positiv_min.append({
                'Stue': stue,
                'Min_Forsinkelse': min_forsinkelse,
                'Gennemsnit': subset['Forsinkelse (minutter)'].mean(),
                'Antal_Operationer': len(subset)
            })

# 4. Lav resultatet om til en DataFrame så det er nemt at læse
df_problem_stuer = pd.DataFrame(stuer_med_positiv_min)

# Sorter så dem med den højeste minimumsforsinkelse står øverst
if not df_problem_stuer.empty:
    df_problem_stuer = df_problem_stuer.sort_values(by='Min_Forsinkelse', ascending=False)
    print("Stuer hvor ALLE operationer kl. 08:00 er forsinkede:")
    print(df_problem_stuer)
else:
    print("Ingen stuer har en minimumsforsinkelse over 0.")

Stuer hvor ALLE operationer kl. 08:00 er forsinkede:
                      Stue  Min_Forsinkelse  Gennemsnit  Antal_Operationer
2             GLO Ø SEC-HK             20.0   31.705882                 17
0             RH TMK AMB 2              7.0   12.000000                  4
1             RH TMK AMB 3              6.0   13.666667                  3
4              CKO LEJE 18              3.0    7.166667                  6
5  OPNORD ANGIO 23.231 (H)              2.0    3.666667                  3
3        OPNORD 13.108 (H)              1.0    4.250000                  4
